# MOHIM Full-Song Melody ControlNet

Paper-faithful DiT ControlNet adaptation of *Editing Music with Melody and Text*. The ACE-Step backbone is frozen, its first 12 DiT blocks are copied, top-k stereo CQT is the only melody-control input, and the copied-block residuals enter the backbone through zero-initialized linear layers. The paper's no-masking ablation is used deliberately: the repeated motif condition is never dropped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys

REPOSITORY = 'https://github.com/youhan200203/MOHIM.git'
BRANCH = 'codex/dit-controlnet'
REPO_DIR = Path('/content/MOHIM')
ACESTEP_DIR = Path('/content/ACE-Step-1.5')
ACESTEP_REPOSITORY = 'https://github.com/ace-step/ACE-Step-1.5.git'
ACESTEP_REVISION = '6d467e4b5081ccb0abf1ec1bf4fdf9051a2d34b0'

if not (REPO_DIR / '.git').is_dir():
    !git clone -b "{BRANCH}" "{REPOSITORY}" "{REPO_DIR}"
%cd {REPO_DIR}
!git fetch origin "{BRANCH}"
!git switch "{BRANCH}"
!git pull --ff-only origin "{BRANCH}"

%pip uninstall -y torchao
%pip install -q bitsandbytes librosa scipy soundfile tensorboard

def write_filtered_requirements(source, destination):
    excluded = ('torchao', 'flash-attn')
    lines = source.read_text(encoding='utf-8').splitlines()
    destination.write_text('\n'.join(
        line for line in lines if not any(name in line.lower() for name in excluded)
    ) + '\n', encoding='utf-8')

root_requirements = Path('/tmp/mohim_controlnet_requirements.txt')
write_filtered_requirements(REPO_DIR / 'requirements.txt', root_requirements)
%pip install -q -r {root_requirements}

if not (ACESTEP_DIR / '.git').is_dir():
    !git clone "{ACESTEP_REPOSITORY}" "{ACESTEP_DIR}"
%cd {ACESTEP_DIR}
!git checkout "{ACESTEP_REVISION}"
ace_requirements = Path('/tmp/acestep_controlnet_requirements.txt')
write_filtered_requirements(ACESTEP_DIR / 'requirements.txt', ace_requirements)
%pip install -q -r {ace_requirements}

for source in (str(REPO_DIR), str(ACESTEP_DIR)):
    if source not in sys.path:
        sys.path.insert(0, source)
os.environ['PYTHONPATH'] = os.pathsep.join([str(REPO_DIR), str(ACESTEP_DIR), os.environ.get('PYTHONPATH', '')])

import bitsandbytes as bnb
print('bitsandbytes:', bnb.__version__)
print('MOHIM:', REPO_DIR)
print('ACE-Step:', ACESTEP_DIR)

In [ ]:
import json, gc, shutil
import torch

VERSION = 'v1_dit_controlnet_topk_cqt_12blocks_no_mask'
DATASET_DIR = Path('/content/drive/MyDrive/MOHIM/motif_dataset_mean_centered')
MANIFEST_PATH = DATASET_DIR / 'full_song_cover_nofsq_manifest.json'
CAPTION_CACHE_PATH = DATASET_DIR / 'ace_full_song_caption_cache.json'
TARGET_TENSOR_DIR = Path('/content/drive/MyDrive/MOHIM/full_song_tensors_repeated_motif')
CQT_CACHE_DIR = Path('/content/drive/MyDrive/MOHIM/controlnet_topk_cqt')
TEXT_CACHE_DIR = Path('/content/drive/MyDrive/MOHIM/controlnet_text_conditions')
CHECKPOINT_DIR = Path('/content/drive/MyDrive/MOHIM/checkpoints')
RUN_DIR = Path('/content/drive/MyDrive/MOHIM/controlnet_runs') / VERSION
MODEL_VARIANT = 'base'
MODEL_DIR = CHECKPOINT_DIR / f'acestep-v15-{MODEL_VARIANT}'
SILENCE_LATENT_PATH = MODEL_DIR / 'silence_latent.pt'
DEVICE = 'cuda'
DTYPE = torch.bfloat16
COPY_BLOCKS = 12
BATCH_SIZE = 1
GRADIENT_ACCUMULATION = 8
EPOCHS = 100
LEARNING_RATE = 5e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
VALIDATION_FRACTION = 0.2
SAVE_EVERY = 1
LOG_EVERY = 10
SEED = 42

for path in (CQT_CACHE_DIR, TEXT_CACHE_DIR, RUN_DIR):
    path.mkdir(parents=True, exist_ok=True)
assert MANIFEST_PATH.is_file(), MANIFEST_PATH
assert CAPTION_CACHE_PATH.is_file(), CAPTION_CACHE_PATH
assert SILENCE_LATENT_PATH.is_file(), SILENCE_LATENT_PATH
manifest = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
caption_cache = json.loads(CAPTION_CACHE_PATH.read_text(encoding='utf-8'))['captions']
target_paths = sorted(TARGET_TENSOR_DIR.glob('*.pt'))
assert len(target_paths) == manifest['metadata']['num_samples']
manifest_by_audio = {str(Path(item['audio_path']).resolve()): item for item in manifest['samples']}
print('samples:', len(target_paths), 'copy blocks:', COPY_BLOCKS)

## 1. Paper top-k CQT cache

No occurrence search is performed. Each short motif is repeated to the target duration before stereo 128-bin top-4 CQT extraction.

In [ ]:
from mohim.controlnet import TOPK_CQT_SCHEMA, extract_repeated_topk_cqt

for index, tensor_path in enumerate(target_paths, 1):
    output_path = CQT_CACHE_DIR / tensor_path.name
    if output_path.is_file():
        cached = torch.load(output_path, map_location='cpu', weights_only=True)
        if cached.get('schema') == TOPK_CQT_SCHEMA:
            continue
    target = torch.load(tensor_path, map_location='cpu', weights_only=True)
    audio_path = str(Path(target['metadata']['audio_path']).resolve())
    sample = manifest_by_audio[audio_path]
    duration = target['target_latents'].shape[0] / 25.0
    melody = extract_repeated_topk_cqt(sample['motif_seed_audio'], duration_seconds=duration)
    temporary = output_path.with_suffix('.pt.tmp')
    torch.save({
        'schema': TOPK_CQT_SCHEMA,
        'melody_pitch_indices': melody,
        'motif_seed_audio': sample['motif_seed_audio'],
        'duration_seconds': duration,
    }, temporary)
    temporary.replace(output_path)
    if index % 10 == 0 or index == len(target_paths):
        print(f'[CQT {index}/{len(target_paths)}] {tuple(melody.shape)}')
assert len(list(CQT_CACHE_DIR.glob('*.pt'))) == len(target_paths)
print('[OK] top-k CQT cache:', CQT_CACHE_DIR)

## 2. Text-to-music condition cache

The existing caption and target latent are reused. Cover instructions are not reused; conditions are encoded with ACE-Step's text2music instruction.

In [ ]:
from acestep.constants import SFT_GEN_PROMPT, TASK_INSTRUCTIONS
from acestep.training.dataset_builder_modules.preprocess_encoder import run_encoder
from acestep.training.dataset_builder_modules.preprocess_lyrics import encode_lyrics
from acestep.training.dataset_builder_modules.preprocess_text import encode_text
from acestep.training_v2.model_loader import load_decoder_for_training, load_text_encoder, unload_models

TEXT_SCHEMA = 'controlnet_text2music_caption_condition_v1'

def build_text2music_prompt(caption, duration):
    metas = f'- bpm: N/A\n- timesignature: N/A\n- keyscale: N/A\n- duration: {duration:.1f} seconds\n'
    return SFT_GEN_PROMPT.format(TASK_INSTRUCTIONS['text2music'], caption, metas)

pending = []
for tensor_path in target_paths:
    output_path = TEXT_CACHE_DIR / tensor_path.name
    if output_path.is_file():
        cached = torch.load(output_path, map_location='cpu', weights_only=True)
        if cached.get('schema') == TEXT_SCHEMA and 'encoder_hidden_states' in cached:
            continue
    pending.append(tensor_path)
print('text conditions to encode:', len(pending))

if pending:
    tokenizer, text_encoder = load_text_encoder(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
    for index, tensor_path in enumerate(pending, 1):
        target = torch.load(tensor_path, map_location='cpu', weights_only=True)
        sample = manifest_by_audio[str(Path(target['metadata']['audio_path']).resolve())]
        caption = caption_cache[str(sample['track_id'])]['caption']
        duration = target['target_latents'].shape[0] / 25.0
        prompt = build_text2music_prompt(caption, duration)
        text_hs, text_mask = encode_text(text_encoder, tokenizer, prompt, DEVICE, DTYPE)
        lyric_hs, lyric_mask = encode_lyrics(text_encoder, tokenizer, sample['lyrics'], DEVICE, DTYPE)
        raw_path = (TEXT_CACHE_DIR / tensor_path.name).with_suffix('.raw.pt')
        torch.save({
            'text_hidden_states': text_hs.cpu(), 'text_attention_mask': text_mask.cpu(),
            'lyric_hidden_states': lyric_hs.cpu(), 'lyric_attention_mask': lyric_mask.cpu(),
        }, raw_path)
        if index % 25 == 0 or index == len(pending):
            print(f'[TEXT PASS 1 {index}/{len(pending)}]')
    unload_models(text_encoder)
    del text_encoder, tokenizer
    gc.collect(); torch.cuda.empty_cache()

    condition_model = load_decoder_for_training(CHECKPOINT_DIR, MODEL_VARIANT, device=DEVICE, precision='bf16')
    for index, tensor_path in enumerate(pending, 1):
        output_path = TEXT_CACHE_DIR / tensor_path.name
        raw_path = output_path.with_suffix('.raw.pt')
        raw = torch.load(raw_path, map_location='cpu', weights_only=True)
        encoder_hs, encoder_mask = run_encoder(
            condition_model,
            raw['text_hidden_states'].to(DEVICE, DTYPE), raw['text_attention_mask'].to(DEVICE),
            raw['lyric_hidden_states'].to(DEVICE, DTYPE), raw['lyric_attention_mask'].to(DEVICE),
            DEVICE, DTYPE,
        )
        temporary = output_path.with_suffix('.pt.tmp')
        torch.save({
            'schema': TEXT_SCHEMA,
            'encoder_hidden_states': encoder_hs.squeeze(0).cpu(),
            'encoder_attention_mask': encoder_mask.squeeze(0).cpu(),
        }, temporary)
        temporary.replace(output_path)
        raw_path.unlink()
        if index % 25 == 0 or index == len(pending):
            print(f'[TEXT PASS 2 {index}/{len(pending)}]')
    unload_models(condition_model)
    del condition_model
    gc.collect(); torch.cuda.empty_cache()

assert len(list(TEXT_CACHE_DIR.glob('*.pt'))) == len(target_paths)
print('[OK] text2music condition cache:', TEXT_CACHE_DIR)

## 3. Load the frozen 24-block ACE DiT and clone its first 12 blocks

In [ ]:
from torch.utils.data import DataLoader
from mohim.controlnet import AceStepDiTControlNet, make_silence_context
from mohim.controlnet_training import (
    ControlNetTensorDataset, collate_controlnet_batch,
    move_batch, split_paths,
)

LOCAL_ROOT = Path('/content/mohim_controlnet_data')
LOCAL_TARGET_DIR = LOCAL_ROOT / 'targets'
LOCAL_CQT_DIR = LOCAL_ROOT / 'cqt'
LOCAL_TEXT_DIR = LOCAL_ROOT / 'text'
if LOCAL_ROOT.exists():
    shutil.rmtree(LOCAL_ROOT)
shutil.copytree(TARGET_TENSOR_DIR, LOCAL_TARGET_DIR)
shutil.copytree(CQT_CACHE_DIR, LOCAL_CQT_DIR)
shutil.copytree(TEXT_CACHE_DIR, LOCAL_TEXT_DIR)
local_paths = sorted(LOCAL_TARGET_DIR.glob('*.pt'))
train_paths, validation_paths = split_paths(local_paths, validation_fraction=VALIDATION_FRACTION, seed=SEED)
train_dataset = ControlNetTensorDataset(LOCAL_TARGET_DIR, LOCAL_CQT_DIR, LOCAL_TEXT_DIR, paths=train_paths)
validation_dataset = ControlNetTensorDataset(LOCAL_TARGET_DIR, LOCAL_CQT_DIR, LOCAL_TEXT_DIR, paths=validation_paths)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, collate_fn=collate_controlnet_batch)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, collate_fn=collate_controlnet_batch)
def load_notebook_silence_latent(path):
    value = torch.load(path, map_location='cpu', weights_only=True)
    if isinstance(value, dict):
        tensors = [item for item in value.values() if torch.is_tensor(item)]
        if len(tensors) != 1:
            raise ValueError(f'Could not identify silence tensor in {path}')
        value = tensors[0]
    if value.ndim == 2:
        value = value.unsqueeze(0)
    if value.ndim == 3 and value.shape[-1] == 64:
        return value.contiguous()
    if value.ndim == 3 and value.shape[1] == 64:
        return value.transpose(1, 2).contiguous()
    raise ValueError(f'Unexpected silence latent shape: {tuple(value.shape)}')

silence_latent = load_notebook_silence_latent(SILENCE_LATENT_PATH)
print('silence latent:', tuple(silence_latent.shape))

loaded_model = load_decoder_for_training(CHECKPOINT_DIR, MODEL_VARIANT, device=DEVICE, precision='bf16')
base_decoder = loaded_model.decoder
loaded_model.decoder = torch.nn.Identity()
del loaded_model
gc.collect(); torch.cuda.empty_cache()
controlnet = AceStepDiTControlNet(base_decoder, copy_blocks=COPY_BLOCKS, gradient_checkpointing=True).to(DEVICE, DTYPE)
controlnet.train()
print('frozen blocks:', len(controlnet.base_decoder.layers))
print('copied trainable blocks:', len(controlnet.control_blocks))
print('trainable parameters:', f'{controlnet.trainable_parameter_count()/1e6:.1f}M')
print('train/validation:', len(train_dataset), len(validation_dataset))

## 4. L4 memory smoke test

This performs a real AdamW8bit optimizer step so its state is included. It never reduces `COPY_BLOCKS=12` automatically.

In [ ]:
from mohim.controlnet_training import memory_smoke_test

smoke_batch = move_batch(next(iter(train_loader)), DEVICE, DTYPE)
smoke_result = memory_smoke_test(
    controlnet, smoke_batch, silence_latent,
    timestep_mu=controlnet.base_decoder.config.timestep_mu,
    timestep_sigma=controlnet.base_decoder.config.timestep_sigma,
    learning_rate=LEARNING_RATE,
)
print(json.dumps(smoke_result, indent=2))
assert smoke_result.get('peak_allocated_gib', 0) < 22.0, smoke_result

# The smoke step changed the copied weights. Recreate all 12 copies from the pristine frozen backbone.
base_decoder = controlnet.base_decoder
controlnet.base_decoder = torch.nn.Identity()
del controlnet, smoke_batch
gc.collect(); torch.cuda.empty_cache()
controlnet = AceStepDiTControlNet(base_decoder, copy_blocks=COPY_BLOCKS, gradient_checkpointing=True).to(DEVICE, DTYPE)
controlnet.train()
print('[OK] 12-block ControlNet memory smoke test passed and weights were reset')

## 5. Train only the melody encoder, copied blocks, and zero linears

In [ ]:
from torch.utils.tensorboard import SummaryWriter
from mohim.controlnet_training import (
    create_adamw8bit, create_inverse_lr, flow_matching_step,
    load_training_checkpoint, save_training_checkpoint, write_history,
)

LATEST_PATH = RUN_DIR / 'latest.pt'
BEST_CONTROL_PATH = RUN_DIR / 'best_controlnet.pt'
HISTORY_PATH = RUN_DIR / 'history.json'
optimizer = create_adamw8bit(controlnet, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = create_inverse_lr(optimizer, inverse_gamma=10_000, power=0.5)
start_epoch, global_step, best_validation_loss, history = 1, 0, float('inf'), []
if LATEST_PATH.is_file():
    state = load_training_checkpoint(LATEST_PATH, model=controlnet, optimizer=optimizer, scheduler=scheduler)
    start_epoch = int(state['epoch']) + 1
    global_step = int(state['global_step'])
    best_validation_loss = float(state['best_validation_loss'])
    history = list(state['history'])
    print('resume epoch:', start_epoch)
writer = SummaryWriter(log_dir=str(RUN_DIR / 'tensorboard'))

for epoch in range(start_epoch, EPOCHS + 1):
    controlnet.train()
    optimizer.zero_grad(set_to_none=True)
    train_sum = 0.0
    for batch_index, cpu_batch in enumerate(train_loader, 1):
        batch = move_batch(cpu_batch, DEVICE, DTYPE)
        with torch.autocast(device_type='cuda', dtype=DTYPE):
            loss = flow_matching_step(
                controlnet, batch, silence_latent,
                timestep_mu=controlnet.base_decoder.config.timestep_mu,
                timestep_sigma=controlnet.base_decoder.config.timestep_sigma,
            )
        (loss / GRADIENT_ACCUMULATION).backward()
        train_sum += float(loss.detach())
        should_step = batch_index % GRADIENT_ACCUMULATION == 0 or batch_index == len(train_loader)
        if should_step:
            torch.nn.utils.clip_grad_norm_(list(controlnet.trainable_parameters()), MAX_GRAD_NORM)
            optimizer.step(); optimizer.zero_grad(set_to_none=True); scheduler.step()
            global_step += 1
            writer.add_scalar('loss/train_step', float(loss.detach()), global_step)
            writer.add_scalar('learning_rate', scheduler.get_last_lr()[0], global_step)
        if batch_index % LOG_EVERY == 0:
            print(f'Epoch {epoch}/{EPOCHS}, batch {batch_index}/{len(train_loader)}, loss={float(loss):.4f}')
        del batch, cpu_batch, loss

    controlnet.eval()
    validation_sum = 0.0
    with torch.no_grad():
        for cpu_batch in validation_loader:
            batch = move_batch(cpu_batch, DEVICE, DTYPE)
            with torch.autocast(device_type='cuda', dtype=DTYPE):
                validation_loss = flow_matching_step(
                    controlnet, batch, silence_latent,
                    timestep_mu=controlnet.base_decoder.config.timestep_mu,
                    timestep_sigma=controlnet.base_decoder.config.timestep_sigma,
                )
            validation_sum += float(validation_loss)
            del batch, cpu_batch, validation_loss
    train_loss = train_sum / len(train_loader)
    validation_loss = validation_sum / len(validation_loader)
    row = {'epoch': epoch, 'train_loss': train_loss, 'validation_loss': validation_loss}
    history.append(row)
    writer.add_scalar('loss/train_epoch', train_loss, epoch)
    writer.add_scalar('loss/validation', validation_loss, epoch)
    writer.flush()
    write_history(HISTORY_PATH, history)
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        torch.save(controlnet.control_state_dict(), BEST_CONTROL_PATH)
    if epoch % SAVE_EVERY == 0:
        save_training_checkpoint(
            LATEST_PATH, model=controlnet, optimizer=optimizer, scheduler=scheduler,
            epoch=epoch, global_step=global_step, best_validation_loss=best_validation_loss, history=history,
        )
    print(f'[OK] Epoch {epoch}/{EPOCHS}, train={train_loss:.4f}, validation={validation_loss:.4f}, best={best_validation_loss:.4f}')
writer.close()

## 6. Custom-motif inference

The same repeated top-k CQT condition is supplied to both CFG branches; CFG therefore scales text only.

In [ ]:
import soundfile as sf

CUSTOM_MOTIF_PATH = Path('/content/motif.wav')
CUSTOM_CAPTION = 'Paste an ACE-Step full-song caption here.'
CUSTOM_LYRICS = 'Paste the complete lyrics here.'
DURATION_SECONDS = 180.0
INFERENCE_STEPS = 50
CFG_SCALE = 7.0
CONTROL_SCALE = 1.0
INFERENCE_SEED = 42
OUTPUT_PATH = RUN_DIR / 'controlnet_generated.wav'
assert CUSTOM_MOTIF_PATH.is_file()
assert not CUSTOM_CAPTION.startswith('Paste')
assert not CUSTOM_LYRICS.startswith('Paste')

# Free training state before loading text encoders.
del optimizer, scheduler
base_decoder = controlnet.base_decoder
controlnet.base_decoder = torch.nn.Identity()
del controlnet, base_decoder
gc.collect(); torch.cuda.empty_cache()

tokenizer, text_encoder = load_text_encoder(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
prompt = build_text2music_prompt(CUSTOM_CAPTION, DURATION_SECONDS)
text_hs, text_mask = encode_text(text_encoder, tokenizer, prompt, DEVICE, DTYPE)
lyric_hs, lyric_mask = encode_lyrics(text_encoder, tokenizer, CUSTOM_LYRICS, DEVICE, DTYPE)
unload_models(text_encoder); del text_encoder, tokenizer
gc.collect(); torch.cuda.empty_cache()

loaded_model = load_decoder_for_training(CHECKPOINT_DIR, MODEL_VARIANT, device=DEVICE, precision='bf16')
encoder_hs, encoder_mask = run_encoder(
    loaded_model, text_hs.to(DEVICE, DTYPE), text_mask.to(DEVICE),
    lyric_hs.to(DEVICE, DTYPE), lyric_mask.to(DEVICE), DEVICE, DTYPE,
)
null_condition = loaded_model.null_condition_emb.detach()
base_decoder = loaded_model.decoder
loaded_model.decoder = torch.nn.Identity()
del loaded_model, text_hs, text_mask, lyric_hs, lyric_mask
gc.collect(); torch.cuda.empty_cache()
controlnet = AceStepDiTControlNet(base_decoder, copy_blocks=COPY_BLOCKS, gradient_checkpointing=False).to(DEVICE, DTYPE).eval()
controlnet.load_control_state_dict(torch.load(BEST_CONTROL_PATH, map_location='cpu', weights_only=False))
melody = extract_repeated_topk_cqt(CUSTOM_MOTIF_PATH, duration_seconds=DURATION_SECONDS).unsqueeze(0).to(DEVICE)
silence_latent = load_notebook_silence_latent(SILENCE_LATENT_PATH)

@torch.inference_mode()
def sample_controlnet():
    latent_length = round(DURATION_SECONDS * 25)
    generator = torch.Generator(device=DEVICE).manual_seed(INFERENCE_SEED)
    latents = torch.randn(1, latent_length, 64, generator=generator, device=DEVICE, dtype=DTYPE)
    context = make_silence_context(silence_latent, batch_size=1, target_length=latent_length, device=DEVICE, dtype=DTYPE)
    audio_mask = torch.ones(1, latent_length, device=DEVICE, dtype=DTYPE)
    condition = encoder_hs.to(DEVICE, DTYPE)
    condition_mask = encoder_mask.to(DEVICE)
    unconditional = null_condition.to(DEVICE, DTYPE).expand_as(condition)
    times = torch.linspace(1.0, 0.0, INFERENCE_STEPS + 1, device=DEVICE, dtype=DTYPE)
    for index in range(INFERENCE_STEPS):
        timestep = times[index].expand(2)
        velocity = controlnet(
            hidden_states=torch.cat([latents, latents]),
            timestep=timestep, timestep_r=timestep,
            attention_mask=torch.cat([audio_mask, audio_mask]),
            encoder_hidden_states=torch.cat([condition, unconditional]),
            encoder_attention_mask=torch.cat([condition_mask, condition_mask]),
            context_latents=torch.cat([context, context]),
            melody_pitch_indices=torch.cat([melody, melody]),
            control_scale=CONTROL_SCALE,
        )[0]
        conditional, unconditional_velocity = velocity.chunk(2)
        guided = unconditional_velocity + CFG_SCALE * (conditional - unconditional_velocity)
        latents = latents + (times[index + 1] - times[index]) * guided
    return latents.cpu()

generated_latents = sample_controlnet()
print('[OK] generated latents:', tuple(generated_latents.shape))

In [ ]:
import math
import torch.nn.functional as F
from IPython.display import Audio, display
from acestep.training_v2.model_loader import load_vae

del controlnet, base_decoder, encoder_hs, encoder_mask, null_condition, melody
gc.collect(); torch.cuda.empty_cache()

def decode_latents_tiled(vae, latents_btc, chunk_frames=256, overlap=64):
    latents = latents_btc.transpose(1, 2)
    stride = chunk_frames - 2 * overlap
    decoded, factor = [], None
    for index in range(math.ceil(latents.shape[-1] / stride)):
        core_start = index * stride
        core_end = min(core_start + stride, latents.shape[-1])
        window_start = max(0, core_start - overlap)
        window_end = min(latents.shape[-1], core_end + overlap)
        chunk = latents[:, :, window_start:window_end].to(DEVICE, dtype=vae.dtype)
        with torch.inference_mode():
            audio = vae.decode(chunk).sample
        factor = factor or audio.shape[-1] / chunk.shape[-1]
        trim_start = round((core_start - window_start) * factor)
        trim_end = round((window_end - core_end) * factor)
        end = audio.shape[-1] - trim_end if trim_end else audio.shape[-1]
        decoded.append(audio[:, :, trim_start:end].float().cpu())
    return torch.cat(decoded, dim=-1)

vae = load_vae(CHECKPOINT_DIR, device=DEVICE, precision='bf16')
audio = decode_latents_tiled(vae, generated_latents)
target_samples = round(DURATION_SECONDS * 48_000)
audio = F.pad(audio, (0, max(0, target_samples - audio.shape[-1])))[:, :, :target_samples]
audio = audio / audio.abs().amax().clamp_min(1.0)
sf.write(OUTPUT_PATH, audio.squeeze(0).transpose(0, 1).numpy(), 48_000)
unload_models(vae); del vae
gc.collect(); torch.cuda.empty_cache()
print(OUTPUT_PATH)
display(Audio(filename=str(OUTPUT_PATH)))